# Aegis-UAV-5G — Colab demo

**Reproducible Agentic AI Testbed for Detection, Attribution and Containment of Cyberattacks in 5G-Enabled UAV Networks.**

This notebook runs the full Level-1 pipeline end-to-end and renders the manuscript tables and figures inline:

1. Setup (install the package)
2. Simulate a mission + inject an attack
3. Quick end-to-end run of the five-agent pipeline (ADA → TCA → AAA → RSA → PEA)
4. Visualise detection, the confusion matrix and latency
5. (Optional) full `smoke` campaign → all Tables 2–6 + Figs 4/5
6. Manuscript data map (`[DATA REQUIRED]` → value → evidence)

> The attack engine is **simulation-only**: every attack is a numeric perturbation of synthetic signals used to train/evaluate the defence.

## 1. Setup

Installs the `aegis-uav-5g` package. The cell auto-detects three situations:
already inside the repo, a clonable GitHub checkout, or an uploaded zip.

In [ ]:
import os, sys, subprocess, glob

def _has_pkg_dir(p): return os.path.isdir(os.path.join(p, 'src', 'aegis_uav'))

root = None
# (a) already inside the aegis-uav-5g directory or the parent repo
for cand in ['.', 'aegis-uav-5g', 'bnt/aegis-uav-5g']:
    if _has_pkg_dir(cand):
        root = cand; break

# (b) try cloning the public repo
if root is None:
    try:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/omega2417/bnt.git', '_bnt'], check=True)
        if _has_pkg_dir('_bnt/aegis-uav-5g'):
            root = '_bnt/aegis-uav-5g'
    except Exception as e:
        print('git clone failed (private repo?):', e)

# (c) fall back to an uploaded zip named aegis-uav-5g*.zip
if root is None:
    zips = glob.glob('aegis-uav-5g*.zip') + glob.glob('*/aegis-uav-5g*.zip')
    if zips:
        import zipfile
        with zipfile.ZipFile(zips[0]) as z: z.extractall('_deposit')
        hits = [os.path.dirname(p) for p in glob.glob('_deposit/**/src/aegis_uav',
                                                       recursive=True)]
        if hits: root = hits[0]

assert root is not None, (
    'Could not locate the package. Upload the Zenodo zip (aegis-uav-5g*.zip) via the',
    'Colab Files panel, or clone the repo, then re-run this cell.')
os.chdir(root)
print('Using package at:', os.getcwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Installed aegis-uav-5g')

## 2. Simulate a mission and inject a GPS-spoofing attack (T1)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from aegis_uav.config import load_scenario, load_attack
from aegis_uav.simulation.scenario_engine import simulate_mission

scenario = load_scenario('configs/scenarios/smoke.yaml')
attack = load_attack('configs/attacks/T1.yaml')
mission = simulate_mission(scenario, attack, seed=0, mission_index=0)
print('mission:', mission.scenario_id, '| rows:', mission.frame.shape[0])

tgt = mission.frame[mission.frame['uav_index'] == attack.target_uavs[0]]
disagreement = np.hypot(tgt['position_x'] - tgt['gnss_reported_x'],
                        tgt['position_y'] - tgt['gnss_reported_y'])
plt.figure(figsize=(8, 3))
plt.plot(tgt['timestamp'].values, disagreement.values)
plt.axvspan(attack.onset_s, attack.onset_s + attack.duration_s, color='red', alpha=0.15,
            label='attack window')
plt.xlabel('time (s)'); plt.ylabel('GNSS vs true position (m)')
plt.title('T1 GPS spoofing — position disagreement on the targeted UAV'); plt.legend()
plt.show()

## 3. Quick end-to-end run of the agentic pipeline (single seed)

Builds a small dataset, trains the agents and runs detection → attribution → response. Takes ~20–40 s on Colab.

In [ ]:
from aegis_uav.config import load_experiment
from aegis_uav.experiments.pipeline import run_core

exp = load_experiment('configs/experiments/smoke.yaml')
exp.dataset.missions_per_class = 3   # keep the demo fast
res = run_core(scenario, exp, seed=0)

print('feature dimension d =', res.feature_dim)
det = pd.DataFrame(res.detection).T[['precision','recall','f1','fpr','auroc']]
det.index.name = 'method'
display(det.round(3))

## 4. Visualise attribution (confusion matrix) and detection latency

In [ ]:
from aegis_uav import ALL_LABELS
cm = res.confusion.astype(float)
norm = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(ALL_LABELS)), ALL_LABELS, rotation=45, ha='right')
ax.set_yticks(range(len(ALL_LABELS)), ALL_LABELS)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Leaf-level attribution')
for i in range(norm.shape[0]):
    for j in range(norm.shape[1]):
        ax.text(j, i, f'{norm[i,j]:.2f}', ha='center', va='center',
                color='white' if norm[i,j] > 0.5 else 'black', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); plt.show()

print('Response (utility-RSA):')
r = res.response.get('utility_rsa', {})
for k in ('n_attack_incidents','contained_before_impact_rate','harmful_response_rate',
          'unnecessary_response_rate','escalation_rate','origin_accuracy'):
    print(f'  {k}: {r.get(k)}')

## 5. (Optional) Full `smoke` campaign — all Tables 2–6 and Figs 4/5

Runs every experiment (E1–E7) across 3 seeds (~6–8 min on Colab) and writes machine-readable tables/figures to `artifacts/`.

In [ ]:
RUN_FULL = False  # set to True to run the full campaign
if RUN_FULL:
    !aegis campaign --config configs/experiments/smoke.yaml
    from IPython.display import Image, Markdown, display
    display(Markdown(open('artifacts/tables/smoke/table_3_detection.md').read()))
    display(Markdown(open('artifacts/tables/smoke/table_5_containment.md').read()))
    display(Image('artifacts/figures/smoke/fig_4_confusion_matrix.png'))
    display(Image('artifacts/figures/smoke/fig_5_latency.png'))
else:
    print('Set RUN_FULL = True to run the full campaign and render all tables/figures.')

## 6. Manuscript data map

Maps each manuscript `[DATA REQUIRED]` item to its computed value and the evidence file it came from (run after a campaign).

In [ ]:
import os
from IPython.display import Markdown, display
if os.path.exists('artifacts/tables/smoke/manuscript_data_map.md'):
    display(Markdown(open('artifacts/tables/smoke/manuscript_data_map.md').read()))
else:
    print('Run the full campaign (cell 5 with RUN_FULL=True), or:')
    print('  !aegis manuscript-map --run-group smoke')

---
Reproduce the full publication campaign locally with:
```bash
aegis campaign --config configs/experiments/paper_v1.yaml
```
See `README.md` and `docs/DATA_AVAILABILITY.md` for details. License: MIT.